# 8주차 실습 — SQL (2) 집계와 JOIN

- **데이터**: `aact_mini` — 7주차와 같은 임상시험 37,509건입니다. 오늘은 `studies` 에 `countries` · `browse_conditions` · `facilities` 를 연결해 봅니다.
- **오늘 배우는 것**
  1. COUNT의 세 가지 모양을 구분해서 쓸 수 있다
  2. GROUP BY로 묶어서 집계하고 HAVING으로 거를 수 있다
  3. CASE WHEN으로 구간을 나누어 세고, COALESCE로 빈 값을 드러낼 수 있다
  4. 두 표를 JOIN으로 연결하고, 행 수가 변하는 것을 확인할 수 있다
- **사용법**: 따라 하기(1~4절, 3절에 절 순서 맞추기 `O1`) → 자주 하는 실수(5절) → 직접 해보기 8문항. 각 문항 아래 `check(...)` 를 실행하면 정답 여부가 바로 표시됩니다.
- **점수는 없습니다.** 7주차 노트북과 같은 방식으로 진행합니다.

## 0. 시작하기 전에

- **Colab 에서 열었다면 먼저 메뉴의 `파일 → Drive에 사본 저장`** 을 누르세요. 그래야 작성한 내용이 남습니다.
- 셀 실행: 셀 왼쪽의 ▶ 단추 또는 `Shift + Enter`. **위에서부터 차례로** 실행합니다.
- `%%sql` 로 시작하는 셀에는 SQL만 적습니다. `%%sql` 바로 아래의 `-- Q1` 같은 줄은 지우지 마세요 (문항 이름표입니다).
- 오류가 나면 붉은 글씨 중 **`Error:` 가 들어간 줄**만 읽으면 됩니다. 연결이 이상하면 `런타임 → 런타임 다시 시작` 후 준비 셀부터 다시 실행하세요.

In [ ]:
#@title ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 학번 셀과 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "바로 위 SQL 셀에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 SQL을 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, f"결과가 표가 아닙니다. %%sql 셀을 실행한 뒤, 바로 다음 셀에서 check('{name}', _) 를 실행하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 check 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

In [ ]:
#@title 자동 확인 도우미 — 정답 SQL 은 들어 있지 않습니다 (수정하지 마세요) { display-mode: "form" }
ANSWERS = _load_answers("eNqdlt9v2lYUx/+VK14CU1XZYGOYtIeuZW2kNNnSZFM1TZHBZo0UJRFQ7aGKRDKnIwlToYVAJsjoSgaNmEIDaahGN2n7T/boe/kfes613diMTEufuD73nou/n/PLj3xzou9j8siXXv4Wfn2ylBSS0Whc1xXZd4341mx7MKlJkYgoRtSghPbU2ndpsMuwTKyt4PJr3/oDNa3jpvmqz3JV3zewfLC8mkF/1tikuyXCcjVm1HB1LzYTu7lA/nn8lHw2P3eXL766E5uP8dXt+bnFz8mn9/nDnRtfTs/e5su5+Vuxecc+M313eoGww226l6N7zeuEVTrMOCGsabC6QUblKt2q0icGoW+GrJUltFUwew3CnnWZMWA/la5zgSlNT+kavGMm9VBHw3pmeW1VXQFLUl1J6xtg+8LLSI7LcUXXQqqouRmN221GopsRaxjstI+7dK9t9gx2mGP1vOv5l2165EV3c25xdsHPKv0Aq2eJ2X3K6n1QnUOKo/0d2spfI9aZW9P3FqZngapzGFk/rxGAQ48GlmsVfmoQHXo0JGx7x0bnJcF1X4Yi6EaRCMuKFBUToi4rbhTj9kkoRk92zPMG7rJmkZUP4PU8ukXWyJFRscYqBdBBrOPkr3Py/jhG2TzrYKCRRbnt6Or+wcoG7fXtODu+7LBAf+2wg+zFFbbN7EJ6dEs0m8dEYeVdxu+EB54//e2rcwq5OUV0OSQrQSku66qb07jd4SR4cmavxg6LrNKFd7mkvOhWg7VqqN08H9B8lke985rVhvRlH8jUMFMwb5DVizaXW3jp1M392I15v3VFAD2tv4LEIvQtcDLsqyZWzX8ykNwMokEhIUYiiiJonlyJa5KiiqGopMRFF4OgG4F5XqPH7UvET4mCQI+3CT0ZwDtPYeKjpdcHcRDfIqY/5Aet/e7qFbT0ilh+eN65AhCx7zenCDt+e3H2CoJlt2AprkU1QU9qESnqFhyKhgVFVAVNCkXdQfcqPvsNosSrY682qlT/FXEIyX4fC4MZEPXTrtkb+nkbCEDBEMgBjDp73kVgoJD3Vow7Vgboqg/tvoqnWWnT4WIOuqyZRY+oIICvjeSDaITdNEJyWBUTmipIeshDY8w+sWvWh7Q3uAREsYBdAiO8msgsLWuEp/DhD2z3zBHFUVje2DsrHfpzB5Wa/azdXVjLwF5aqNITXjzWJsyrqpeZdQuhp7lRdcB7TbNIe2dw52N+NZTIlatEcWOKh4KhkJaIy0LIM4DH7RM7KtT4OB3jGcbyfRvlQ9jSYM1HLq+VGx1UUSzUz2hz6BDC+1wV0y2Zf5ZYeQieDaufQGc4bmMAUDTfcPhZWPgJa/hCT+XD1xk5xM/7O89YdIexlOVD+rjN9ouBq/CLeNIsHI5LybCqRbSIJ83G7BPTbHKCXSCE6WlN2o8CCNJvk/y7YiMMEJb/EU/BHe7vEusY8gWvsVk92mpaTfdDhnHULV1UlGRQ0MNyOKK7pY/bJ0mnrw27wcIKZqKHgNVdMFPM0zyIoQWnnMgnduuxd6BQ/Sk9/XAlk15KLqfSmaX1tXRG15Y0NaMHsELoGwP6Fn6/OJMWv1Ms8A4x4lfYiyHrtsns4szM/8gF/HTb2HgH3Ni7hw==")

## 1. 집계 함수 — COUNT · SUM · AVG · MEDIAN

여러 행을 하나의 숫자로 만듭니다. 다섯 개면 대부분의 통계가 됩니다.

| 함수 | 무엇을 구하나 | 예 | NULL은 |
|---|---|---|---|
| `COUNT` | 개수 | `COUNT(*)` | `COUNT(*)`는 포함, `COUNT(열)`은 제외 |
| `SUM` | 합계 | `SUM(enrollment)` | 제외하고 더함 |
| `AVG` | 평균 | `AVG(enrollment)` | 제외하고 나눔 |
| `MIN` / `MAX` | 가장 작은 / 큰 값 | `MAX(start_date)` | 제외 |
| `MEDIAN` | 중앙값 | `MEDIAN(enrollment)` | 제외 |

집계 함수는 NULL을 빼고 계산합니다. 그래서 '빠진 값'이 많으면 결과가 달라집니다.

### COUNT — 세 가지 모양

| 쓰는 법 | 무엇을 세나 | studies에서 | 주의 |
|---|---|---:|---|
| `COUNT(*)` | 모든 행 | 37,509 | NULL도 포함해서 셈 |
| `COUNT(열)` | 그 열에 값이 있는 행 | 4,352 | NULL은 세지 않음 |
| `COUNT(DISTINCT 열)` | 서로 다른 값의 개수 | 9 | 중복을 뺀 종류 수 |

In [ ]:
%%sql
-- COUNT 세 가지 모양
SELECT COUNT(*) AS 전체행,
       COUNT(results_first_posted_date) AS 결과있음,
       COUNT(DISTINCT overall_status) AS 상태종류
FROM studies;

'몇 건인가'를 물을 때는 **무엇을 세는지** 먼저 말로 정하세요.

### 평균과 중앙값 — 평균 하나만 보면 안 되는 이유

중재 연구(INTERVENTIONAL) 27,916건의 등록 인원을 봅시다. 전체와, 1만 명 이하만 따로 계산합니다.

In [ ]:
%%sql
-- 전체 → 평균 1,279 / 중앙값 60 / 최대 12,317,546
SELECT ROUND(AVG(enrollment)) AS 평균,
       MEDIAN(enrollment) AS 중앙값,
       MAX(enrollment) AS 최대
FROM studies
WHERE study_type = 'INTERVENTIONAL';

In [ ]:
%%sql
-- 1만 명 이하만 → 평균 196 / 중앙값 60 / 최대 10,000
SELECT ROUND(AVG(enrollment)) AS 평균,
       MEDIAN(enrollment) AS 중앙값,
       MAX(enrollment) AS 최대
FROM studies
WHERE study_type = 'INTERVENTIONAL'
  AND enrollment <= 10000;

1,200만 명짜리 입력 오류처럼 아주 큰 값(1만 명이 넘는 시험 142건)이 섞여 있어, 평균이 중앙값의 20배가 됩니다. 중앙값은 60명으로 거의 변하지 않습니다.

> **병원 데이터라면** — 재원일수 · 진료비 · 검사 수치도 한쪽으로 치우친 값이 대부분입니다. 평균과 중앙값을 함께 보고, 이상한 값은 원인을 확인한 뒤 판단합니다.

### 비율 구하기 — 퍼센트는 100.0 을 곱해서 계산합니다

In [ ]:
%%sql
-- “완료된 시험 중 결과가 게시된 비율은?” → 18.0
SELECT ROUND(100.0
         * COUNT(results_first_posted_date)
         / COUNT(*), 1) AS 결과게시율
FROM studies
WHERE overall_status = 'COMPLETED';

100 이 아니라 **100.0** 을 곱하는 이유: 정수끼리 나누면 소수점이 잘려 0이 나옵니다. (12주차 오류)

> 참고: 이 실습의 DuckDB 는 정수끼리 나누어도 소수로 계산해 줍니다. PostgreSQL 같은 다른 DB 에서는 소수점이 잘리므로(예: 3524 / 19622 → 0) 100.0 을 곱하는 습관을 들입니다.

## 2. GROUP BY — 묶어서 세기

같은 값끼리 묶어서 셉니다. **결과 한 행 = 그룹 하나** 입니다.

In [ ]:
%%sql
-- “연구 유형별로 몇 건씩 있어?”
SELECT study_type, COUNT(*) AS 건수,
       MEDIAN(enrollment) AS 중앙값
FROM studies
GROUP BY study_type
ORDER BY 건수 DESC;

묶는 열(`study_type`)은 SELECT에도 적어야 결과를 읽을 수 있습니다.

### 연도별 추이 — 날짜에서 연도만 뽑아 묶습니다

In [ ]:
%%sql
-- “연도별로 시작한 시험이 몇 건이야?”
SELECT YEAR(start_date) AS 시작연도,
       COUNT(*) AS 건수
FROM studies
GROUP BY 시작연도
ORDER BY 시작연도;

`YEAR(날짜)` = 날짜에서 연도만 꺼내는 함수. 월은 `MONTH(날짜)`, 연월은 `strftime(날짜, '%Y-%m')` 입니다.

### 두 열로 묶기 — 교차표 만들기

In [ ]:
%%sql
-- “최근 2년, 연도별 · 연구 유형별 건수는?”
SELECT YEAR(start_date) AS 연도,
       study_type AS 유형,
       COUNT(*) AS 건수
FROM studies
WHERE start_date >= '2023-01-01'
GROUP BY 연도, 유형
ORDER BY 연도, 유형;

묶는 열을 여러 개 적으면 결과 한 행은 '연도 + 유형' 조합 하나가 됩니다.

### 빈 값도 드러내기 — COALESCE

NULL 은 이름을 붙여야 보입니다.

In [ ]:
%%sql
-- “임상 단계별 건수를 보여 줘. 안 적힌 것도 같이.”
SELECT COALESCE(phase, '미기재') AS 단계,
       COUNT(*) AS 건수
FROM studies
GROUP BY 단계
ORDER BY 건수 DESC;

`phase` 에는 빈 값(NULL) 9,602건과 글자 `'NA'` 16,695건이 따로 있습니다. 둘은 다른 값입니다.

### CASE WHEN — 조건에 따라 나누어 세기

엑셀의 IF 와 같은 일을 합니다.

In [ ]:
%%sql
-- “등록 인원을 소 · 중 · 대규모로 나누어 세어 줘”
SELECT CASE WHEN enrollment < 50
            THEN '소규모'
            WHEN enrollment < 500
            THEN '중규모'
            ELSE '대규모'
       END AS 규모,
       COUNT(*) AS 건수
FROM studies
WHERE enrollment IS NOT NULL
GROUP BY 규모;

위에서부터 차례로 맞는 조건을 찾고, 맞으면 거기서 멈춥니다. 어디에도 안 맞으면 `ELSE`.

> **병원 데이터라면** — CASE WHEN 을 이렇게 씁니다.
>
> | 하려는 일 | 이렇게 적습니다 |
> |---|---|
> | 나이를 연령대로 묶기 | `CASE WHEN 나이 < 40 THEN '39세 이하' WHEN 나이 < 65 THEN '40~64세' ELSE '65세 이상' END` |
> | 검사값을 정상 · 이상으로 | `CASE WHEN 수치 BETWEEN 하한 AND 상한 THEN '정상' ELSE '이상' END` |
> | 조건에 맞는 것만 세기 | `COUNT(CASE WHEN 재입원 = 'Y' THEN 1 END)` |
> | 분모와 분자를 한 번에 | `COUNT(*) AS 분모, COUNT(CASE WHEN 충족 = 'Y' THEN 1 END) AS 분자` |

## 3. HAVING과 실행 순서 — 집계 결과에 조건

| 절 | 하는 일 | 예 | 실제 처리 순서 |
|---|---|---|:---:|
| `SELECT` | 볼 열과 집계를 고른다 | `SELECT phase, COUNT(*)` | ⑤ |
| `FROM` | 어느 표에서 가져올지 정한다 | `FROM studies` | ① |
| `WHERE` | 행을 걸러낸다 (묶기 전) | `WHERE enrollment >= 100` | ② |
| `GROUP BY` | 같은 값끼리 묶는다 | `GROUP BY phase` | ③ |
| `HAVING` | 묶은 결과를 걸러낸다 | `HAVING COUNT(*) >= 10` | ④ |
| `ORDER BY` | 정렬한다 | `ORDER BY COUNT(*) DESC` | ⑥ |
| `LIMIT` | 앞에서 몇 줄만 | `LIMIT 5` | ⑦ |

적는 순서와 실행 순서가 다릅니다. 그래서 WHERE 에는 집계 함수를 쓸 수 없습니다. 집계에 대한 조건은 WHERE가 아니라 **HAVING**에 씁니다.

### 절 순서 맞추기 — 직접 해 보기

**O1.** “2024년에 시작한 시험을 임상 단계별로 세고, 많은 순으로 위에서 5줄만 보고 싶다.”

아래 셀의 여섯 조각은 슬라이드처럼 순서가 뒤섞여 있습니다. 먼저 그대로 실행해 붉은 글씨 중 `Error:` 가 들어간 줄(`Parser Error: syntax error at or near "LIMIT"`)을 보고, **줄의 순서만 바꾸어** 다시 실행한 뒤 `check` 를 실행하세요.

적는 순서: `SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT`

In [ ]:
%%sql
-- O1
-- [순서 맞추기] 순서가 뒤섞여 있어 그대로 실행하면 문법 오류가 납니다. 줄의 순서만 바꾸세요.
LIMIT 5
GROUP BY phase
SELECT phase, COUNT(*) AS 건수
ORDER BY 건수 DESC
FROM studies
WHERE start_date >= '2024-01-01'

In [ ]:
check("O1", _)

순서를 바꿔 적으면 문법 오류가 납니다. 외우기보다 이 줄을 옆에 적어 두고 쓰세요.

### HAVING — WHERE는 묶기 전, HAVING은 묶은 뒤

In [ ]:
%%sql
-- “시험이 2000건 이상인 국가만 보여 줘”
SELECT name AS 국가,
       COUNT(DISTINCT nct_id) AS 시험수
FROM countries
WHERE removed = false
GROUP BY name
HAVING COUNT(DISTINCT nct_id) >= 2000
ORDER BY 시험수 DESC;

`WHERE removed = false` 는 묶기 전에 제외된 국가 행을 빼는 조건입니다.

## 4. JOIN — 두 표를 연결하기

정규화로 나눈 표를 다시 붙입니다.

### 실습 DB는 이미 정규화되어 있습니다

6주차에 배운 정규화를 실제 표에서 확인합니다.

![aact_mini 표 구조 한눈에 보기](https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini_tables.png)

7주차에 본 표 구조 그림입니다. 모든 표가 `nct_id` 로 `studies` 와 이어집니다.

| 테이블 | 한 행이 뜻하는 것 | 행 수 | 왜 따로 나눴나 |
|---|---|---:|---|
| `studies` | 임상시험 1건 | 37,509 | 제목 · 단계 · 상태는 시험마다 한 번만 저장 |
| `browse_conditions` | 시험 × 질환 1쌍 | 309,124 | 한 시험에 질환이 여러 개 (한 칸에 값 하나, 1NF) |
| `countries` | 시험 × 국가 1쌍 | 50,410 | 한 시험이 여러 나라에서 진행 |

한 표에 모두 담으면 시험 제목이 질환 수만큼 반복됩니다. 나눠 둔 표를 필요할 때 다시 붙이는 것이 JOIN입니다.

### JOIN이 필요한 이유 — 한 표에 없는 정보를 다른 표에서 가져오기

`studies` 표에는 국가 정보가 없습니다. 국가는 `countries` 표에 따로 있습니다.
두 표 모두 `nct_id` 를 가지고 있어서, 이 열로 같은 시험끼리 맞춰 붙입니다. 두 표를 잇는 조건은 **ON** 절에 적습니다.

In [ ]:
%%sql
-- studies 와 countries 를 nct_id 로 연결
SELECT s.nct_id, s.brief_title, c.name AS 국가
FROM studies AS s
JOIN countries AS c ON s.nct_id = c.nct_id

표 이름을 `s` · `c` 처럼 짧게 붙여 두면(별칭) 어느 표의 열인지 분명해집니다.

### 질환별 시험 수 구하기

In [ ]:
%%sql
-- “뇌졸중 · 고혈압 · 천식은 각각 시험이 몇 건이야?”
SELECT b.mesh_term AS 질환,
       COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
WHERE b.mesh_term IN ('Stroke', 'Hypertension', 'Asthma')
GROUP BY b.mesh_term
ORDER BY 시험수 DESC;

질환 이름은 `conditions`(연구자 입력)가 아니라 `browse_conditions` 를 씁니다. MeSH = 의학 주제어 표준 목록. 같은 질환을 한 이름으로 부르기 위한 것입니다 (14주차 표준 용어).

### JOIN의 두 가지 — 양쪽에 다 있는 것만 vs 왼쪽은 모두

studies 37,509건 중 2,699건은 기관(facilities) 정보가 없습니다.

| JOIN (= INNER JOIN) | LEFT JOIN |
|---|---|
| 양쪽에 모두 있는 행만 남김 | 왼쪽(studies)은 모두 남김. 오른쪽에 없으면 NULL로 채움 |
| 결과: 34,810건 (기관 정보가 없는 시험은 빠짐) | 결과: 37,509건 그대로 |

In [ ]:
%%sql
-- JOIN → 34,810건
SELECT COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN facilities AS f ON s.nct_id = f.nct_id;

In [ ]:
%%sql
-- LEFT JOIN → 37,509건
SELECT COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
LEFT JOIN facilities AS f ON s.nct_id = f.nct_id;

“전체 시험 중 몇 %”를 구할 때 INNER JOIN을 쓰면 **분모가 조용히 줄어듭니다.** 12주차에서 가장 자주 나오는 오류입니다.

### JOIN 뒤에는 행 수를 확인 — 한 시험에 질환이 여러 개 붙어 있습니다

In [ ]:
%%sql
-- “JOIN하면 행이 몇 개가 되지?”
SELECT COUNT(*) AS JOIN후_행수,
       COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b
  ON s.nct_id = b.nct_id;

행이 약 10배로 늘었습니다. 이 상태에서 `COUNT(*)`를 세면 시험 수가 아니라 (시험 × 질환) 쌍의 수입니다.

### 반정규화 실습 — 넓은 표 만들어 보기

나눠 둔 표를 JOIN으로 합쳐 한 표로 저장합니다. `OR REPLACE` 를 붙여 다시 실행해도 오류가 나지 않게 했습니다.

In [ ]:
%%sql
-- “시험과 질환을 한 표로 합쳐 둬”
CREATE OR REPLACE TEMP TABLE 시험_질환 AS
SELECT s.nct_id, s.brief_title,
       s.phase, b.mesh_term
FROM studies AS s
JOIN browse_conditions AS b
  ON s.nct_id = b.nct_id;

In [ ]:
%%sql
-- 만든 표 확인 → 309,124행 · 31,634건 (바로 위 JOIN 결과와 같습니다)
SELECT COUNT(*) AS 행수,
       COUNT(DISTINCT nct_id) AS 서로다른_시험
FROM 시험_질환;

JOIN 없이 바로 꺼내 쓸 수 있어 편하지만, 제목 · 단계가 질환 수만큼 반복됩니다. 11주차에 볼 CDW의 넓은 표가 이런 모양입니다.

TEMP 표는 노트북을 닫으면 사라집니다. 런타임을 다시 시작했다면 위의 `CREATE OR REPLACE TEMP TABLE 시험_질환` 셀부터 다시 실행하세요.

### 넓은 표의 문제 — 6주차 이상현상 세 가지

`시험_질환` 에서 시험 하나를 골라 봅니다. 같은 제목이 질환 수만큼 반복됩니다.

In [ ]:
%%sql
-- 시험 하나가 질환 수만큼 여러 행
SELECT nct_id, brief_title, mesh_term
FROM 시험_질환
WHERE nct_id = 'NCT04110262';

| 질문 | 답 | 이상현상 |
|---|---|---|
| ① 시험 하나의 제목을 고치려면 몇 행을 바꿔야 할까? | 그 시험의 질환 수만큼. 한 행이라도 빠뜨리면 제목이 서로 달라짐 | 갱신 이상 |
| ② 질환 정보가 없는 시험은 이 표에 있을까? | 없다. 37,509건 중 5,875건이 빠짐. 이 표만 남기면 그 시험이 사라짐 | 삭제 이상 |
| ③ 질환이 정해지지 않은 새 시험을 넣을 수 있을까? | `mesh_term` 을 비워 둔 채 넣어야 함. 질환이 정해질 때까지 넣기 곤란 | 삽입 이상 |

①과 ②는 아래 직접 해보기 Q7 · Q8 에서 SQL 로 세어 봅니다.

운영 DB는 나눠서(정규화) 저장하고, 분석용 CDW는 합쳐서(반정규화) 씁니다. 넓은 표에서는 `COUNT(DISTINCT 키)` 로 셉니다.

## 5. 자주 하는 실수 — 행 수를 꼭 확인

`[오류 예시]` 셀을 실행해 오류 메시지(또는 결과)를 보고, 바로 아래의 고친 SQL과 비교하세요. 오류 메시지는 붉은 글씨 중 `Error:` 가 들어간 줄(예: `Binder Error: …`)만 읽으면 됩니다. **오류가 안 나는 실수가 더 위험합니다.**

### 실수 1 — GROUP BY에 없는 열 (실행하면 오류 메시지가 나옵니다)

In [ ]:
%%sql
-- [오류 예시] 이렇게 쓰면 → Binder Error
SELECT phase, overall_status,
       COUNT(*)
FROM studies
GROUP BY phase

In [ ]:
%%sql
-- 이렇게 고칩니다
SELECT phase, overall_status,
       COUNT(*)
FROM studies
GROUP BY phase, overall_status

묶는 열이 아니면 집계 함수로 감싸거나 GROUP BY에 넣습니다.

### 실수 2 — WHERE에 집계 함수 (실행하면 오류 메시지가 나옵니다)

In [ ]:
%%sql
-- [오류 예시] 이렇게 쓰면 → Binder Error (WHERE는 묶기 전이라 COUNT를 모릅니다)
SELECT phase, COUNT(*)
FROM studies
WHERE COUNT(*) > 1000
GROUP BY phase

In [ ]:
%%sql
-- 이렇게 고칩니다
SELECT phase, COUNT(*)
FROM studies
GROUP BY phase
HAVING COUNT(*) > 1000

집계에 대한 조건은 HAVING에 씁니다.

### 실수 3 — JOIN 뒤 COUNT(*)를 시험 수로 착각 (오류가 나지 않아 더 위험합니다)

In [ ]:
%%sql
-- [오류 예시] 결과 4,098 — 오류는 없지만 이것은 기관 수입니다
SELECT COUNT(*) AS 한국시험수
FROM studies AS s
JOIN facilities AS f
  ON s.nct_id = f.nct_id
WHERE f.country = 'South Korea'

In [ ]:
%%sql
-- 이렇게 고칩니다 → 1,182 (실제 시험 수)
SELECT COUNT(DISTINCT s.nct_id)
         AS 한국시험수
FROM studies AS s
JOIN facilities AS f
  ON s.nct_id = f.nct_id
WHERE f.country = 'South Korea'

한 시험이 여러 기관에서 진행되기 때문입니다. `COUNT(DISTINCT 키)`를 씁니다.

**집계와 JOIN을 쓸 때의 확인 습관** — 이 네 가지만 지켜도 대부분의 오류를 막습니다.
1. 무엇을 세는지 말로 정한다 — “시험 건수”인가 “기관 수”인가 “질환 쌍의 수”인가
2. JOIN 전후의 행 수를 비교한다 — `COUNT(*)` 와 `COUNT(DISTINCT 키)` 를 같이 본다
3. 분모가 줄지 않았는지 본다 — INNER JOIN 때문에 빠진 행이 있는지 LEFT JOIN으로 확인
4. 평균은 중앙값과 함께 본다 — 극단값 하나가 결과를 바꿀 수 있다

## 직접 해보기 — 8문항

각 문항의 `%%sql` 셀에서 `___` 를 채워 실행한 뒤, **바로 다음 셀**의 `check(...)` 를 실행하세요. 두 번 틀리면 힌트가 나옵니다.
결과 열의 이름(`AS ...`)은 적힌 그대로 둡니다.

**Q1. COUNT 세 가지** — 임상 단계(`phase`)에 **값이 있는**(NULL이 아닌) 시험 수(`단계있음`, 글자 'NA' 도 값입니다)와, `phase` 값의 **종류 수**(`단계종류`)를 세세요.

In [ ]:
%%sql
-- Q1
SELECT COUNT(*) AS 전체,
       COUNT(___) AS 단계있음,
       COUNT(___) AS 단계종류
FROM studies;

In [ ]:
check("Q1", _)

> 7주차 Q8 의 DISTINCT 는 9가지였는데 여기서는 8 입니다. COUNT(DISTINCT 열)은 빈 값(NULL)을 세지 않기 때문입니다.

**Q2. 평균과 중앙값** — 3상(`PHASE3`) 시험의 등록 인원 평균(`평균`, 반올림)과 중앙값(`중앙값`)을 구하세요. 두 값이 얼마나 다른지 보세요.

In [ ]:
%%sql
-- Q2
SELECT ROUND(___(enrollment)) AS 평균,
       ___(enrollment) AS 중앙값
FROM studies
WHERE phase = 'PHASE3';

In [ ]:
check("Q2", _)

> 평균이 중앙값의 4배가 넘습니다. 아주 큰 시험 몇 건이 평균을 끌어올렸기 때문입니다.

**Q3. GROUP BY** — 완료(`COMPLETED`)된 시험을 **시작 연도별로** 세세요. 결과 한 행 = 연도 하나입니다. (2절 '연도별 추이' 예제를 참고하세요)

In [ ]:
%%sql
-- Q3
SELECT ___ AS 시작연도, COUNT(*) AS 건수
FROM studies
WHERE overall_status = 'COMPLETED'
GROUP BY ___
ORDER BY 시작연도;

In [ ]:
check("Q3", _)

**Q4. CASE WHEN** — 등록 인원을 '100명 미만'과 '100명 이상' 두 구간으로 나누어 세세요. `___` 에 조건을 적습니다.

In [ ]:
%%sql
-- Q4
SELECT CASE WHEN ___ THEN '100명 미만'
            ELSE '100명 이상'
       END AS 규모,
       COUNT(*) AS 건수
FROM studies
WHERE enrollment IS NOT NULL
GROUP BY 규모;

In [ ]:
check("Q4", _)

**Q5. HAVING** — 시험이 **900건 이상**인 국가만 보세요. `___` 자리에 조건을 한 줄로 적습니다. WHERE와 HAVING 중 어느 것일까요?

In [ ]:
%%sql
-- Q5
SELECT name AS 국가,
       COUNT(DISTINCT nct_id) AS 시험수
FROM countries
WHERE removed = false
GROUP BY name
___
ORDER BY 시험수 DESC;

In [ ]:
check("Q5", _)

**Q6. JOIN** — 일본(`Japan`) 기관에서 진행된 **시험 수**를 세세요. 두 표를 잇는 조건(ON)과 세는 방법을 채웁니다. (5절 실수 3을 참고하세요)

In [ ]:
%%sql
-- Q6
SELECT COUNT(___) AS 일본시험수
FROM studies AS s
JOIN facilities AS f ON ___
WHERE f.country = 'Japan';

In [ ]:
check("Q6", _)

**Q7. 넓은 표 ① 갱신 이상** — 시험 `NCT02369328` 의 제목을 고치려면 `시험_질환` 에서 몇 행을 바꿔야 할까요? 이 시험이 `시험_질환` 에 몇 행 있는지(`행수`) `___` 에 세는 식을 적으세요. (`시험_질환` 이 없다는 오류가 나면 4절의 `CREATE OR REPLACE TEMP TABLE 시험_질환` 셀을 먼저 실행하세요)

In [ ]:
%%sql
-- Q7
SELECT ___ AS 행수
FROM 시험_질환
WHERE nct_id = 'NCT02369328';

In [ ]:
check("Q7", _)

> 결과의 행 수만큼 제목을 모두 바꿔야 합니다. 한 행이라도 빠뜨리면 같은 시험의 제목이 서로 달라집니다 → **갱신 이상**.

**Q8. 넓은 표 ② 삭제 이상** — `시험_질환` 에 들어 있는 **서로 다른 시험 수**(`시험수`)를 세세요. `studies` 의 37,509건과 비교합니다.

In [ ]:
%%sql
-- Q8
SELECT COUNT(___) AS 시험수
FROM 시험_질환;

In [ ]:
check("Q8", _)

> 37,509 − 31,634 = **5,875건**은 질환 정보(`browse_conditions`)가 없어 이 표에 들어오지 못했습니다. `시험_질환` 만 남기고 나눠 둔 표를 지우면 이 시험들이 사라집니다 → **삭제 이상**.

### 더 해보기 (선택)

**Q9. 분모와 분자를 한 번에** — 완료된 시험 수(`분모`)와, 그중 결과가 게시된 시험 수(`분자`)를 한 번에 세세요. 분자 ÷ 분모가 1절의 결과게시율 18.0% 와 같은지 확인해 보세요.

In [ ]:
%%sql
-- Q9
SELECT COUNT(*) AS 분모,
       COUNT(CASE WHEN ___ THEN 1 END) AS 분자
FROM studies
WHERE overall_status = 'COMPLETED';

In [ ]:
check("Q9", _)

## 오늘 배운 것

- GROUP BY 결과의 한 행이 무엇인지 말로 확인한다
- WHERE는 묶기 전, HAVING은 묶은 뒤 조건이다
- CASE WHEN으로 구간을 나누고, COALESCE로 빈 값에 이름을 붙인다
- JOIN 뒤에는 행 수가 변한다. COUNT(DISTINCT 키)로 확인한다

다음 시간: 9강. 의료데이터 SQL 실습 — 분석 질문을 SQL로 바꾸기, 과제 1 제출

아래 셀을 실행하면 오늘 확인한 문항(O1, Q1~Q8, 선택 Q9)을 한눈에 볼 수 있습니다.

In [ ]:
진행확인()